# Classify the given image as a cat or a dog
- Download the dataset from : https://www.dropbox.com/scl/fi/ppd8g3d6yoy5gbn960fso/dataset.zip?dl=0&e=2&rlkey=lqbqx7z6i9hp61l6g731wgp4v&st=gdn6pydw

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Counting the no of images present in the test and train set

In [2]:
import os
print(f"No of dog images in test set: {len(os.listdir("../datasets/dataset/test_set/dogs/"))}")
print(f"No of cat images in test set: {len(os.listdir("../datasets/dataset/test_set/cats/"))}")
print(f"No of dog images in train set: {len(os.listdir("../datasets/dataset/training_set/dogs/"))}")
print(f"No of cat images in test set: {len(os.listdir("../datasets/dataset/training_set/cats/"))}")

No of dog images in test set: 1001
No of cat images in test set: 1001
No of dog images in train set: 4001
No of cat images in test set: 4001


# Preprocessing on the training set

- We apply transformations (like flips, zoomin zoom out, rotate) on the training set to avoid overfitting 
- This process is called as the image augmentation
- Note data augmentation is applied only on train dataset
- But scaling and resizing must be applied on the test dataset too

In [3]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range = 0.2,
    zoom_range= 0.2,
    horizontal_flip= True
)

# Data augmentation on the training set
training_set = train_datagen.flow_from_directory(
    "../datasets/dataset/training_set/",
    target_size= (64,64), # Final size of the images
    batch_size=32, # How many images must be processed in each batch
    class_mode = "binary"
)

# Scaling, Resizing on test set
test_datagen = ImageDataGenerator(
    rescale = 1./255
)

testing_set = test_datagen.flow_from_directory(
    "../datasets/dataset/test_set/",
    target_size = (64,64),
    batch_size = 32,
    class_mode = "binary"
)


Found 8000 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.


- It implies there are total of 8000 images in train set and 2000 images in test set with 2 classes each i.e. dog and cat

# Building the CNN

In [4]:
model = tf.keras.models.Sequential()


# Adding layer to the CNN : Convolution Layer
- In the very first layer you need to provide the input shape of the image

In [5]:
# Adding the Convolution layer 
model.add(tf.keras.layers.Conv2D(
    filters = 32, # Standard value (copied from the famous architecture, you can play around with different architecture's configurations)
    kernel_size = 3,
    activation = "relu",
    input_shape = (64,64,3) # Resized shape and as its a coloured image 3 channels
))


/Users/ritumalage/Documents/my_github/knowlegebase/.venv/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


# Pooling Layer

In [6]:
model.add(tf.keras.layers.MaxPool2D(
    pool_size = 2,# Kernel size that must be used over the image
    strides = 2, # By how many pixels should the kernel be shifted by
))

# Adding 2nd Convolution Layer

In [7]:
# Adding the Convolution layer 
model.add(tf.keras.layers.Conv2D(
    filters = 32, # Standard value (copied from the famous architecture, you can play around with different architecture's configurations)
    kernel_size = 3,
    activation = "relu",
))


# Adding 2nd pooling layer

In [8]:
model.add(tf.keras.layers.MaxPool2D(
    pool_size = 2,# Kernel size that must be used over the image
    strides = 2, # By how many pixels should the kernel be shifted by
))

# Adding the Flattening layer
- No parameters is passed to the Flatten Layer

In [9]:
model.add(tf.keras.layers.Flatten())

# Adding the fully connected layer

In [10]:
model.add(tf.keras.layers.Dense(
    units = 128, # No of neurons 
    activation = "relu"
))

# Output Layer
- No of units is 1 as its a binary classification problem, else will be equal to the no of classes present
- Activation Function in the output layer must not be RELU, as its a binary classification set it to sigmoid else for multi class classification set it to softmax

In [11]:
model.add(tf.keras.layers.Dense(
    units = 1,
    activation = "sigmoid"
))

# Early Stopping
- This ensures training is stopped automatically when the validation loss starts increasing

In [12]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor = "val_loss",
    patience = 5, # Wait until 5 epochs before stopping, even after 5 epochs if the val loss does not improve then the training stops
    restore_best_weights = True 
)

# Training
- We train on test set and evaluate it on test set

In [13]:
model.compile(
    optimizer = "adam",
    loss = "binary_crossentropy",
    metrics = ["accuracy"]
)

model.fit(
    x = training_set,
    validation_data = testing_set,
    epochs = 100,
    callbacks = [early_stopping]
    
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 8s 32ms/step - accuracy: 0.5805 - loss: 0.6713 - val_accuracy: 0.6510 - val_loss: 0.6254
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - accuracy: 0.6645 - loss: 0.6120 - val_accuracy: 0.7140 - val_loss: 0.5740
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step - accuracy: 0.7013 - loss: 0.5629 - val_accuracy: 0.7270 - val_loss: 0.5478
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step - accuracy: 0.7324 - loss: 0.5316 - val_accuracy: 0.7435 - val_loss: 0.5280
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step - accuracy: 0.7508 - loss: 0.5088 - val_accuracy: 0.7465 - val_loss: 0.5251
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step - accuracy: 0.7591 - loss: 0.4930 - val_accuracy: 0.7590 - val_loss: 0.4921
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 26ms/step - accuracy: 0.7746 - loss: 0.4719 - val_accuracy: 0.7455 - val_loss: 0.5343
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - accuracy: 0.7800 - loss: 0.4633 - 

# Making a single prediction
- The new unseen image must be resized to the same size that was used during the training process here it is 64
- `.predict()` method expects a 2D array of the image which can be done using 
```
from keras.preprocessing import image
test_image = image.img_to_array(test_image)
```
- While training we had specified the no of batches as 32, now even while testing on the test dataset new dimension must be added which acts as no of batches
`test_image = np.expand_dims(test_image, axis = 0)`
- Now the unseen image is brought into the format required
`result = model.predict(test_image)`

# Fetch the label information

In [14]:
training_set.class_indices

{'cats': 0, 'dogs': 1}